# MiniLab en Colab
Ejecuta las celdas en orden. Antes: **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.

Los resultados y puntos de guardado quedan en tu Drive, carpeta `minilab_results`. Si Colab corta la sesión, ejecuta de nuevo las celdas 1, 2 y 4: continúa donde iba.

In [ ]:
# 1. GPU, Drive y repo
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
from google.colab import drive; drive.mount('/content/drive')
%cd /content
!test -d minilab || git clone -q https://github.com/Cristian-CTG/minilab
%cd /content/minilab
!git pull -q
!test -d minimind || git clone -q --depth 1 https://github.com/jingyaogong/minimind
!mkdir -p /content/drive/MyDrive/minilab_results data
!test -L results || (rm -rf results && ln -s /content/drive/MyDrive/minilab_results results)
!pip -q install tokenizers
!ls results | head

In [ ]:
# 2. Datos (unos 10-15 min la primera vez de cada sesión)
!test -f data/pretrain_t2t_mini.jsonl || curl -L -o data/pretrain_t2t_mini.jsonl https://huggingface.co/datasets/jingyaogong/minimind_dataset/resolve/main/pretrain_t2t_mini.jsonl
!test -f data/train.json || python lab/prep.py 1000000
!test -f data/test.json || python lab/prep_test.py
!cat data/train.json data/val.json

In [ ]:
# 3. Prueba de velocidad: bf16 contra fp16 (2-3 min). Anota cuál da más tok_per_s y úsalo en la celda 4
%cd /content/minilab/lab
for amp in ('bf16', 'fp16'):
    print(amp)
    !python -u train.py bench-{amp} opt=muon amp={amp} steps=60 eval_every=60 2>&1 | tail -1
%cd /content/minilab

In [ ]:
# 4. Cola de escalado (12 entrenamientos). Cambia AMP según la celda 3
AMP = 'fp16'
!AMP={AMP} bash lab/queue_colab.sh

In [ ]:
# 5. Resumen (el archivo completo está en Drive: minilab_results/runs.jsonl)
import json
for l in open('results/runs.jsonl'):
    r = json.loads(l)
    print(f"{r['name']:16s} semilla {r['run']['seed']}  {r['params']/1e6:5.1f}M  bpb {r['bpb']:.4f}  {r['secs']/60:5.1f} min  {r.get('machine','')}")